# 1 · The matplotlib Mental Model
*Data Visualization for Scientists & Public Health Professionals*

Almost every chart you make in Python is drawn by **matplotlib** — directly, or under the hood by seaborn and pandas. Before any specific chart type, it pays to learn the small object hierarchy the whole library rests on: a **Figure** that holds one or more **Axes**, each of which is a single plot you label, size, and save. Get this model right once and most of the rest of the course is vocabulary.

### Learning objectives
- Tell matplotlib's two interfaces apart, and use the object-oriented one by default
- Describe the Figure / Axes hierarchy that underlies every plot
- Label, title, and size a plot through its `Axes`
- Place several plots in one figure
- Save a figure correctly for slides versus print

### Agenda
1. One fact, two charts — why choices matter
2. A quick chart chooser
3. Two interfaces, one library
4. The Figure / Axes hierarchy
5. Anatomy of a plot
6. Several plots in one figure
7. Size, resolution, and saving
8. The bridge to seaborn

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Setup

We use `diabetes_viz` — about 102,000 hospital encounters for diabetic patients, cleaned so we can spend our attention on plotting rather than tidying. We read it from the course repository with the same f-string pattern you used all through the pandas course. We also fix the natural order of the age bands once, so every age plot in the notebook reads youngest-to-oldest instead of alphabetically.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/diabetes_viz.csv", low_memory=False)

# age is an ordered category; reuse this order all notebook
age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
df.shape

## 1. One fact, two charts — why choices matter

Here is a single fact — how many encounters each hospital specialty handled — drawn two ways. Same numbers, same data. One you can read; the other is confetti. That gap, repeated across every decision a chart asks of you, is what this whole course is about: a good chart is a series of deliberate choices, and matplotlib's defaults are rarely the best of them.

In [ ]:
counts = df["medical_specialty"].value_counts()          # 72 distinct specialties

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# left: a pie of all 72 specialties -- every slice correct, the whole thing useless
axes[0].pie(counts.values)
axes[0].set_title(f"A pie of all {len(counts)} specialties")

# right: the ten busiest as a horizontal bar -- readable in one glance
top10 = counts.head(10)
axes[1].barh(top10.index[::-1], top10.values[::-1], color="steelblue")
axes[1].set_title("Ten busiest specialties")
axes[1].set_xlabel("Encounters")

fig.tight_layout()
plt.show()

> **Note:** Nothing about the pie is *wrong* — every wedge is exactly the right size. It is simply unreadable: no one can rank 72 wedges or compare their angles. The bar answers "which specialties dominate?" instantly, because length is easy to compare and the bars are ordered. Same data, better choices — that is the entire job.

## 2. A quick chart chooser

Before drawing anything, ask two questions: *what am I trying to show*, and *what kind of variables do I have?* The answer usually points straight at one chart. Keep this map nearby — the rest of the course fills it in.

| Your question | Variables | Reach for | Covered in |
|---|---|---|---|
| What shape is this variable? | one numeric | histogram, KDE, box, violin | this session |
| How do groups compare? | one categorical, one number | bar / count plot | this session |
| Do two variables move together? | two numeric | scatter, trend line | next session |
| How does something change over time? | numeric over dates | line, rolling average | next session |
| Which of many pairs correlate? | many numeric | correlation heatmap | next session |

Every one of those is drawn on the same Figure/Axes foundation we build right now.

## 3. Two interfaces, one library

matplotlib gives you two ways to draw the same thing.

The **pyplot (state-machine) style** is terse: you call `plt.plot`, `plt.title`, and matplotlib quietly tracks a "current" figure behind the scenes. It reads cleanly for a throwaway plot, but it hides *what* you are drawing on, and it gets awkward the moment a figure holds more than one panel.

In [ ]:
stay_by_age = df.groupby("age")["time_in_hospital"].mean().reindex(age_order)

# State-machine style: short, but you hold no handle on the figure or the plot area
plt.plot(age_order, stay_by_age.values, marker="o")
plt.title("Mean length of stay by age")
plt.show()

The **object-oriented (OO) style** makes the objects explicit: `plt.subplots()` hands you a `Figure` and an `Axes`, and you call methods on the `Axes`. It is a little more typing for one plot and far less for everything after — which is exactly why it is what seaborn hands back, and what almost every example online with an `ax` variable assumes. **We use it from here on.**

In [ ]:
fig, ax = plt.subplots()
ax.plot(age_order, stay_by_age.values, marker="o")
ax.set_title("Mean length of stay by age")
plt.show()

> **Tip:** Coming from MATLAB, R, or SAS? The state-machine style will feel familiar — it is close to MATLAB's plotting model. The OO style is the one worth the muscle memory here, because it scales to the multi-panel figures later in the course without any ambiguity about which plot you are talking to.
>
> [Matplotlib quick-start guide](https://matplotlib.org/stable/users/explain/quick_start.html)

## 4. The Figure / Axes hierarchy

Three words, easy to confuse:

- **Figure** — the whole canvas, the page. It can hold one plot or many.
- **Axes** — a *single plot*: one data area with its own title, ticks, and legend. This is the object you touch most.
- **Axis** — the x or y number line *inside* an Axes. You reach for it rarely (custom ticks, limits).

The naming is unfortunate: an **Axes** is a whole plot, not a line. `plt.subplots()` returns the Figure and its Axes together.

In [ ]:
fig, ax = plt.subplots()
print(type(fig))   # the whole canvas
print(type(ax))    # one plot area inside it
plt.close(fig)     # nothing drawn yet, so close the empty figure

> **Tip:** Whenever an example online opens with `fig, ax = plt.subplots()` and then calls `ax.something(...)`, this hierarchy is what it is leaning on. Learn to read that first line and the rest usually follows.

## 5. Anatomy of a plot

A finished plot is data plus the labels that make it legible, and the `Axes` is where all of it lives: `ax.plot(...)` draws the data, and the `ax.set_*` methods add the title and axis names. A plot without axis labels is not finished — treat labels as part of the deliverable, not an optional polish step.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(age_order, stay_by_age.values, marker="o", color="steelblue", label="Mean stay")

ax.set_title("Length of stay rises with age")
ax.set_xlabel("Age band")
ax.set_ylabel("Mean days in hospital")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()   # keep labels from being clipped
plt.show()

> **Note:** This is a real finding, not a toy: mean length of stay climbs from about 2.6 days for the youngest bands to nearly 4.8 for the oldest — a pattern the overall average would completely hide. Every `set_*` method has a matching getter (`ax.get_title()`), and the full set is large: limits, ticks, scales, spines. [Axes API reference](https://matplotlib.org/stable/api/axes_api.html)

### Exercise 1 — A fully labeled plot *(5 min)*

Compute the **mean `num_lab_procedures` for each age band** and draw it as a labeled line in the object-oriented style: title, x-label, y-label, and a legend. Reuse `age_order` so the bands read youngest-to-oldest. In a comment, note whether lab procedures rise with age as sharply as length of stay did.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
labs_by_age = df.groupby("age")["num_lab_procedures"].mean().reindex(age_order)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(age_order, labs_by_age.values, marker="o", color="steelblue", label="Mean lab procedures")
ax.set_title("Lab procedures by age band")
ax.set_xlabel("Age band")
ax.set_ylabel("Mean lab procedures")
ax.legend()
fig.tight_layout()
plt.show()
# Nearly flat: lab procedures drift only from about 41 to 45 across the whole age range,
# unlike length of stay, which nearly doubled.
```

**Why this works.** `groupby("age").mean()` gives one value per band, and `reindex(age_order)` puts those bands in human order instead of alphabetical. Everything after is the anatomy from the demo: draw on the `Axes`, then label through it. The flat line is itself the answer — age barely moves lab-test volume, even though it strongly moves length of stay.

</details>

## 6. Several plots in one figure

Ask `plt.subplots` for a grid and it returns the Figure plus an **array of Axes**, which you draw on by index. This is where the OO style earns its keep: there is no ambiguous "current" plot to lose track of — each panel is a named object you address directly.

In [ ]:
top_spec = df["medical_specialty"].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))   # 1 row, 2 columns

# left panel: a line
axes[0].plot(age_order, stay_by_age.values, marker="o", color="steelblue")
axes[0].set_title("Mean stay by age")
axes[0].set_xlabel("Age band")
axes[0].set_ylabel("Mean days")
axes[0].tick_params(axis="x", rotation=45)

# right panel: a horizontal bar
axes[1].barh(top_spec.index[::-1], top_spec.values[::-1], color="seagreen")
axes[1].set_title("Eight busiest specialties")
axes[1].set_xlabel("Encounters")

fig.tight_layout()
plt.show()

> **Tip:** For a grid with more than one row *and* column, `axes` is 2-D (`axes[0, 1]`). Pass `sharex=True` or `sharey=True` to lock panels onto a common scale so they are honestly comparable, and `constrained_layout=True` in `subplots(...)` is a modern alternative to `fig.tight_layout()`.

### Exercise 2 — Two panels side by side *(7 min)*

Combine two things you have now seen into one figure with **one row and two columns**. Left panel: the **mean `num_lab_procedures` by age** line from Exercise 1. Right panel: a **bar of encounter counts per `race`** (use `value_counts`). Title and label both panels so a reader can tell at a glance what each one shows.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
labs_by_age = df.groupby("age")["num_lab_procedures"].mean().reindex(age_order)
race_counts = df["race"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(age_order, labs_by_age.values, marker="o", color="steelblue")
axes[0].set_title("Mean lab procedures by age")
axes[0].set_xlabel("Age band")
axes[0].set_ylabel("Mean lab procedures")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(race_counts.index, race_counts.values, color="seagreen")
axes[1].set_title("Encounters by race")
axes[1].set_ylabel("Encounters")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()
```

**Why this works.** Each panel is just another `Axes`, addressed by its index in the returned array, so the line and the bar are built with exactly the tools you already have — they simply live side by side now. `tick_params(rotation=...)` keeps the longer category labels from colliding.

</details>

## 7. Size, resolution, and saving

`figsize=(width, height)` is in **inches**; `dpi` (dots per inch) sets how many pixels those inches become when the figure is rasterized. Size controls the shape and how much room labels get; dpi controls sharpness of the saved image. Save with `fig.savefig(...)` on the Figure object — never a screenshot — and two rules cover almost everything: a **raster** format (`.png`, with a `dpi`) for slides and the web, and a **vector** format (`.pdf` or `.svg`) for print or anything that may be scaled up.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(age_order, stay_by_age.values, marker="o", color="steelblue")
ax.set_title("Mean length of stay by age")
ax.set_xlabel("Age band")
ax.set_ylabel("Mean days in hospital")
fig.tight_layout()

fig.savefig("stay_by_age.png", dpi=150, bbox_inches="tight")   # raster, for slides/web
fig.savefig("stay_by_age.pdf", bbox_inches="tight")            # vector, for print
plt.show()
print("saved stay_by_age.png and stay_by_age.pdf")

> **Tip:** `bbox_inches="tight"` trims the surrounding whitespace, and `transparent=True` helps when a figure will sit on a colored slide. In Colab, saved files land in the file browser on the left (the folder icon) and download from there. [savefig documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html)

## 8. The bridge to seaborn

Most of the rest of this course uses **seaborn**, which draws statistical plots in a single line. The one thing to carry across: seaborn draws onto a matplotlib `Axes`. Pass it `ax=`, and everything you just learned — titles, labels, sizing, saving — still applies, because you are customizing the very same object.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(x=age_order, y=stay_by_age.values, marker="o", sort=False, ax=ax)
ax.set_title("Same line, now drawn by seaborn")
ax.set_xlabel("Age band")
ax.set_ylabel("Mean days in hospital")
fig.tight_layout()
plt.show()

> **Note:** seaborn handles the *drawing*; matplotlib still owns the *figure*. That split is the through-line of the whole course — when a seaborn plot needs a tweak it does not offer, you drop down to the `Axes` and use exactly what you learned here.

## Wrap-up

You now have the model the rest of the course stands on: a **Figure** holds one or more **Axes**; you draw and label through the Axes in the object-oriented style; you size with `figsize`, and you save with `fig.savefig` — raster for slides, vector for print. seaborn rides on top of all of it. And you have already seen the course's real thesis in miniature — the pie versus the bar — that the difference between a confusing chart and a clear one is a handful of deliberate choices.

**Next:** Distributions — histograms, KDEs, boxplots, and violins for seeing the shape of a single variable, and the bin-width choice that can quietly change the story.